## Function Definitions

In [ ]:
using Printf
using LinearAlgebra
using CSV, DataFrames

#=
Make_D1_D2

INPUT:bl::Int, br::Int, wl::Int, wr::Int, r::Int, D1_til::DiagonalMatrix,
D2_til::DiagonalMatrix

GOAL: Build the GSVD matrices D1 and D2 based on the block structure
defined by bl, br, wl, wr, r, and the "central" matrices D1_til and
D2_til.

OUTPUT: D1::Matrix, D2::Matrix

=#

function Make_D1_D2(bl,br,wl,wr,r, D1_til, D2_til)
    D1= [I(bl)        zeros(bl,r) zeros(bl,br);
         zeros(r,bl)  D1_til      zeros(r,br);
         zeros(wl,bl) zeros(wl,r) zeros(wl,br)]

    #D2 in our format
    D2= [zeros(wr,bl) zeros(wr,r) zeros(wr,br);
         zeros(r,bl)  D2_til      zeros(r,br);
         zeros(br,bl) zeros(br,r) I(br)]

    return D1, D2
end

#=
Make_A_B

INPUT:bl::Int, br::Int, wl::Int, wr::Int, r::Int, D1_til::Matrix,
D2_til::Matrix, H::Matrix, U::Matrix, V::Matrix

GOAL: Build A and B from the decomposition produced by the GSVD, using the block
structure defined by bl, br, wl, wr, r, the "central" matrices D1_til and
D2_til (to create D1 and D2), and H (surjective), U and V (orthogonal) to create A and B.

OUTPUT: A::Matrix, B::Matrix

=#

function Make_A_B(bl,br,wl,wr,r,D1_til,D2_til,H,U,V)
    D1,D2 = Make_D1_D2(bl,br,wl,wr,r, D1_til, D2_til)

    A = U * D1 * H
    B = V * D2 * H

    return A, B
end

#=
Our_SVD

INPUT: A::Matrix, B::Matrix

GOAL: Decompose A and B using a modified GSVD, because the D2 returned by Julia in svd(A,B) is
not blocked in the way we defined in Make_D1_D2. For this, the permutation matrix P is used to produce V_til. D2_our is D2 in our blocked format.
For this reason, NEVER USE THE D2 RETURNED DIRECTLY BY JULIA WITHOUT APPLYING THIS PERMUTATION.

OUTPUT: U::Matrix, V_til::Matrix, H::Matrix, D1_til::Matrix, D2_til::Matrix, D1::Matrix, D2_our::Matrix, bl::Int, br::Int, wl::Int, wr::Int, r::Int, P::Matrix, V::Matrix

=#

function Our_SVD(A,B)
    bl,br,wl,wr,r = wire_size(A,B)
    U, V, Q, D1, D2_julia, R0 = svd(A, B)

    P, D2_our = permutation(wr,r,br, D2_julia)
    V_til = V * P #Changing variable to fix line changes in D2

    #Selects D1_tilde and D2_tilde
    D1_til = D1[bl+1:bl+r, bl+1:bl+r]
    D2_til = D2_our[wr+1:wr+r, bl+1:bl+r]

    H = R0 * Q'

    return U, V_til, H, D1_til, D2_til, D1, D2_our, bl, br, wl, wr, r, P, V
end

#=
TO DO: clarify the wire_size and permutation functions with Júlia
=#
function wire_size(A,B)
    m,.. = size(A)
    p,.. = size(B)
    .., .., .., D1, D2, .. = svd(A, B)
    ..,k_mais_l = size(D1)

    #Counting k: D1 is a m-by-(k_mais_l) diagonal matrix with 1s in the first k entries,
    k = 0
    for i in 1:min(m, k_mais_l)
        if D1[i, i] == 1
            k += 1
        else
            k += 0
        end
    end

    l = k_mais_l-k

    #Counting r: D2 is a matrix whose upper-right l-by-l block is diagonal,
    #   with the first r entries belonging to D2_til and the rest 1s.
    r = 0
    for i in 1:l
        if D2[i, k+i] != 1
            r += 1
        else
            r += 0
        end
    end

    bl = k
    br = l - r
    wl = m - k - r
    wr = p - l

    return bl,br,wl,wr,r
end

function permutation(wr,r,br, D2_julia)
    P = [zeros(r,wr)  I(r)        zeros(r,br);
         zeros(br,wr) zeros(br,r) I(br);
         I(wr)        zeros(wr,r) zeros(wr,br)]

    P_til = [zeros(wr,r) zeros(wr,br) I(wr);
             I(r)        zeros(r,br)  zeros(r,wr);
             zeros(br,r) I(br)        zeros(br,wr)]

    D2_our = P_til * D2_julia

    return P, D2_our
end


In [ ]:
using MLDatasets
using Random
using Statistics
using LinearAlgebra
using Plots

function get_MNIST_vectors(digit, k)
    imgs, labels = MNIST(split=:test)[:]

    imgs = Float64.(imgs) ./ 255

    idx = findall(labels .== digit)

    if length(idx) < k
        error("There are not enough images for the chosen digit.")
    end
    idx = Random.shuffle(idx)[1:k]
    X = transpose(hcat([vec(imgs[:, :, i]) for i in idx]...))
    return X
end


function create_MNIST_matrices(digitA, digitB, n, m)

    imgs, labels = MNIST(split=:train)[:]

     # normalize
    imgs = Float64.(imgs) ./ 255 

    # find the digits
    idxA = findall(labels .== digitA)
    idxB = findall(labels .== digitB)

    # handle errors
    if length(idxA) < n || length(idxB) < m
        error("There are not enough images for the chosen digits.")
    end

    # shuffle the indices
    Random.seed!(1234)  
    idxA = Random.shuffle(idxA)[1:n]
    idxB = Random.shuffle(idxB)[1:m]

     # Vectorize and stack as columns
    A = hcat([vec(imgs[:, :, i]) for i in idxA]...)
    B = hcat([vec(imgs[:, :, i]) for i in idxB]...)
    
    # transform into row vectors 
    #A = transpose(hcat([vec(imgs[:, :, i]) for i in idxA]...))
    #B = transpose(hcat([vec(imgs[:, :, i]) for i in idxB]...))

    return A, B
end


function plot_matrix(M, titulo, k=nothing, media=nothing, savepath=nothing, linha_ini=nothing, linha_fim=nothing)
    # We now treat "linha_ini/linha_fim" as a COLUMN RANGE (images)
    n_cols_total = size(M, 2)

    col_ini = linha_ini === nothing ? 1 : linha_ini
    col_fim = linha_fim === nothing ? n_cols_total : linha_fim

    col_ini = clamp(col_ini, 1, n_cols_total)
    col_fim = clamp(col_fim, col_ini, n_cols_total)

    # Column submatrix (images)
    M_exibir = M[:, col_ini:col_fim]

    # If a mean is provided, add it to each column (mean is a 784-vector)
    if media !== nothing
        media_vec = vec(media)  # garante forma (784,)
        @assert length(media_vec) == size(M_exibir, 1) "mean must have length equal to the number of rows (pixels) of M"
        M_exibir = M_exibir .+ media_vec
    end

    # Number of images to plot
    k_imgs = size(M_exibir, 2)
    if k !== nothing
        k_imgs = min(k_imgs, k)
        M_exibir = M_exibir[:, 1:k_imgs]
    end

    max_por_linha = 10
    ncols = min(k_imgs, max_por_linha)
    nrows = ceil(Int, k_imgs / ncols)

    plt = plot(layout = (nrows, ncols), size = (120 * ncols, 120 * nrows), title = titulo)

    for i in 1:k_imgs
        # each column is a 28x28 image
        img = reshape(M_exibir[:, i], (28, 28))
        heatmap!(
            plt[i],
            img[:, end:-1:1]',
            axis = nothing,
            colorbar = false,
            framestyle = :none,
            aspect_ratio = 1,
            c = :viridis
        )
    end

    display(plt)
    if savepath !== nothing
        savefig(plt, savepath)
    end
end

In [ ]:
function get_nonzero_per_column(matrix)
    return [begin
        col = matrix[:, j]
        idx = findfirst(!iszero, col)
        idx !== nothing ? col[idx] : zero(eltype(matrix)) 
    end for j in 1:size(matrix, 2)]
end


function get_angles_D2(D2_our)
    nonzero_values = get_nonzero_per_column(D2_our)
    angles_deg = rad2deg.(asin.(nonzero_values))
    return angles_deg
end

function get_angles_D1(D1)
    nonzero_values = get_nonzero_per_column(D1)
    angles_deg = rad2deg.(acos.(nonzero_values))
    return angles_deg
end

function plot_angles(D)
    nonzero_values = get_nonzero_per_column(D)
    arccos_values_deg = rad2deg.(acos.(nonzero_values))
    
    plot(arccos_values_deg, 
         marker=:circle, 
         xlabel="Index", 
         ylabel="Angle (degrees)", 
         title="Arccosine of the values (in degrees)",
         legend=false,
         grid=true)
    hline!([45], linestyle=:dash, color=:red, label="45°")
end

function highlight_closest(angles_deg, alvo_deg, k=20)
    diffs = abs.(angles_deg .- alvo_deg)

    # indices sorted from closest to farthest
    sorted_all_idxs = sortperm(diffs)
    top_k_idxs = sorted_all_idxs[1:k]

    # reorder the top-k by how close they are to the target
    ordered_idxs = sort(top_k_idxs, by=i -> diffs[i])

    return ordered_idxs
end


In [ ]:
function plot_recon_A(D1, H, i, media_A)
    row = D1[i, :]
    idx = findfirst(!iszero, row)
    idx !== nothing ? img_vec = media_A + D1[i, idx] .* H[:, i] : img_vec = media_A + 0 .* H[:, i]


    # reshape to 28x28
    img = reshape(img_vec, (28, 28))

    # Plotar a imagem
    heatmap(img[:, end:-1:1]',
            axis=nothing,
            colorbar=false,
            framestyle=:none,
            aspect_ratio=1,
            c=:viridis,
            title="Reconstructed image from term $i")       
end

function plot_recon_B(D2_our, H, i, media_B)
    row = D2_our[i, :]
    idx = findfirst(!iszero, row)
    idx !== nothing ? img_vec = media_B + D2_our[i, idx] .* H[:, i] : img_vec = media_B + 0 .* H[:, i]
    

    img = reshape(img_vec, (28, 28))

    heatmap(img[:, end:-1:1]',
            axis=nothing,
            colorbar=false,
            framestyle=:none,
            aspect_ratio=1,
            c=:viridis,
            title="Reconstructed image from term $i")
            
end


In [ ]:
function cosine_similarity(x, y)
    return dot(x, y) / (norm(x) * norm(y))
end

function get_most_similar_row_H(x, H)
    similarity = Vector(undef, size(H, 1))
    for i in 1:size(H, 1)
        similarity[i] = cosine_similarity(x, H[i, :])
    end
    return similarity
end

function plot_vec(x; display=true)

    # reshape to 28x28
    img = reshape(x, (28, 28))

    # Plotar a imagem
    plt = heatmap(img[:, end:-1:1]',
            axis=nothing,
            colorbar=false,
            framestyle=:none,
            aspect_ratio=1,
            c=:viridis,
    )

    if display
        display(plt)
    end
    
    return plt
end

In [ ]:
function sort_and_rebuild(U, V, D1, D2, D1_til, D2_til, H, bl, br, wl, wr, r)
    d1_vals = get_nonzero_per_column(D1_til)
    d2_vals = get_nonzero_per_column(D2_til)

    p = sortperm(d2_vals)

    d1_sorted = d1_vals[p]
    d2_sorted = d2_vals[p]

    H_sorted = copy(H)
    H_sorted[bl+1:bl+r, :] = H[bl+1:bl+r, :][p, :]

    U_sorted = copy(U)
    U_sorted[:, bl+1:bl+r] = U[:, bl+1:bl+r][:,p]

    V_sorted = copy(V)
    V_sorted[:, wr+1:wr+r] = V[:, wr+1:wr+r][:, p]

    D1_til_sorted = Diagonal(d1_sorted)
    D2_til_sorted = Diagonal(d2_sorted)

    D1_sorted = [I(bl)           zeros(bl, r)    zeros(bl, br);
                 zeros(r, bl)    D1_til_sorted   zeros(r, br);
                 zeros(wl, bl)   zeros(wl, r)    zeros(wl, br)]

    D2_sorted = [zeros(wr, bl)  zeros(wr, r)     zeros(wr, br);
                 zeros(r, bl)   D2_til_sorted   zeros(r, br);
                    zeros(br, bl)  zeros(br, r)    I(br)]

    return U_sorted, V_sorted, D1_til_sorted, D2_til_sorted, D1_sorted, D2_sorted, H_sorted
end


In [ ]:
function set_to_0_closest_to_45(D1, D2, k)
    angles = get_angles_D1(D1)

    # find the index of the angle closest to 45° (the central one)
    idx_central = argmin(abs.(angles .- 45))

    D1r = copy(D1)
    D2r = copy(D2)

    # always zero out the central term
    D1r[:, idx_central] .= 0
    D2r[:, idx_central] .= 0

    # split indices on the left (<45) and right (>45)
    left = findall(a -> a < 45, angles)
    right = findall(a -> a > 45, angles)

    # sort from closest to 45 to farthest
    sort!(left, by = i -> -angles[i])
    sort!(right, by = i -> angles[i])

    # remove k symmetric pairs
    for i in 1:min(k, min(length(left), length(right)))
        li = left[i]
        ri = right[i]
        D1r[:, li] .= 0
        D2r[:, li] .= 0
        D1r[:, ri] .= 0
        D2r[:, ri] .= 0
    end

    return D1r, D2r
end


In [ ]:
#D1_red_sym, D2_red_sym = set_to_0_symmetric(D1, D2, 3)

In [ ]:
#display(D1_red_sym)
#display(D2_red_sym)

In [ ]:
#display(D1_red)
#display(D2_red)

In [ ]:
function get_all_MNIST(digit; split=:test)
    imgs, labels = MNIST(split=split)[:] # get all images and labels from the test set
    imgs = Float64.(imgs) ./ 255 # normalize

    idx = findall(labels .== digit) # get the digit indices

    X = hcat([vec(imgs[:, :, i]) for i in idx]...) # place the images as columns

    return X
end

## Point Test

In [ ]:
function plot_points(A, B, Zs)
    scatter(A[1, :], A[2, :],
        label="Subspace A",
        color=:blue,
        marker=:circle,
        legend=:topright)

    scatter!(B[1, :], B[2, :],
        label="Subspace B",
        color=:red,
        marker=:utriangle)

    for (i, Z) in enumerate(Zs)
        scatter!([Z[1]], [Z[2]],
            label="Z_$i",
            marker=:star5,
            ms=10)
    end

    xlabel!("x₁")
    ylabel!("x₂")
    title!("Subspaces A and B (not perpendicular)")
end

In [ ]:
n_points_A = 80
n_points_B = 75

# Subspace A ~ approximately aligned with the x-axis
x_A = 5 .* randn(n_points_A)
y_A = 1 .* randn(n_points_A)
A = hcat(x_A, y_A)'

# Subspace B ~ initially aligned with the y-axis
x_B = 2 .* randn(n_points_B)
y_B = 5 .* randn(n_points_B)
B = hcat(x_B, y_B)'

# Apply a rotation so that it is not perpendicular
θ = π/6  # 30 degrees
R = [cos(θ) -sin(θ); sin(θ) cos(θ)]
B = R * B

# Special points
Z1 = [5.0, 0.05]    # close to the x-axis
Z2 = [0.1, 5.0]     # close to the former y-axis
Z3 = [-0.4, 0.2]         # near the origin
Z4 = [100,100]

Zs = [Z1, Z2, Z3, Z4]
plot_points(A, B, Zs)


In [ ]:
U, V_til, H, D1_til, D2_til, D1, D2_our, bl, br, wl, wr, r_val, P, V = Our_SVD(A', B')

println("U size: ", size(U))
println("V_til size: ", size(V_til))
println("H size: ", size(H))
println("D1 size: ", size(D1))
println("D2_our size: ", size(D2_our))
println("D1_til: ", size(D1_til))
println("D2_til: ", size(D2_til))
println("bl: ", bl)
println("br: ", br)
println("wl: ", wl)
println("wr: ", wr)
println("r_val: ", r_val)
println("u_wires: ", bl+r_val+wl)
println("v_wires: ", wr+r_val+br)

In [ ]:
U_new, V_new, D1_til_sorted, D2_til_sorted, D1_new, D2_our_new, H_new = sort_and_rebuild(
  U, V_til, D1, D2_our, D1_til, D2_til, H, bl, br, wl, wr, r_val
)

println("Dimensions:")
println("U_new: ", size(U_new))
println("V_new: ", size(V_new))
println("D1_til_sorted", size(D1_til_sorted))
println("D2_til_sorted", size(D2_til_sorted))
println("D1_new: ", size(D1_new))
println("D2_new: ", size(D2_our_new)) 
println("H_new: ", size(H_new))

In [ ]:
c = vec(H_new' \ Z1) # solve the least-squares problem: vector c contains the coefficients that best approximate vector v as a linear combination of the columns of matrix H_new'
wA = D1_new * c
wB = D2_our_new* c


classificador = norm(wA)/norm(wB)
println("Classifier (norm WA / norm WB): ", classificador)
# Category for points inside the intersection
if abs(classificador - 1) < 0.2  # Proximity band around the intersection
    println("Classified as in the intersection (point inside the intersection)")
elseif classificador > 1
    println("Classified as digit A")
else
    println("Classified as digit B")
end
energia_c = norm(c)^2
energia_A = norm(wA)^2
energia_B = norm(wB)^2


soma_energias_componentes = energia_A + energia_B

diferenca = energia_c - soma_energias_componentes
println("\n||c||²:           ", energia_c)
println("||w_A||²:           ", energia_A)
println("||w_B||²:           ", energia_B)
println("Difference (Energy_c - Sum):     ", diferenca)

if isapprox(energia_c, soma_energias_componentes)
    println("\n The identity ||c||² ≈ ||w_A||² + ||w_B||² has been verified!")
else
    println("\n X")
end


### ====================================================================================================

## Simple GSVD test of A and B (evaluation on the initially chosen pair)

In [ ]:
digitA = 4
digitB = 9
n = 900
m = 800

A, B = create_MNIST_matrices(digitA, digitB, n, m)

println("Size of matrix A: ", size(A))  # (n, 784)
println("Size of matrix B: ", size(B))  # (m, 784)


### Plot A and B

In [ ]:
plot_matrix(A, "Digit $digitA", 20)

In [ ]:
plot_matrix(B, "Digit $digitB",20)

### Centering

In [ ]:
# centering
media_A = mean(A, dims=2) |> vec  # dims=2 for column-wise mean
media_B = mean(B, dims=2) |> vec # dims=1 for row-wise mean
display(size(media_A))
A = A .- media_A
B = B .- media_B

### Extracting the means of A and B

In [ ]:
plot_vec(media_A)

In [ ]:
plot_vec(media_B)

In [ ]:
plot_matrix(A, "Digit $digitA", 20)

In [ ]:
plot_matrix(B, "Digit $digitB", 20)

### GSVD of A and B

Standard GSVD from Julia


In [ ]:
U, V_til, H, D1_til, D2_til, D1, D2_our, bl, br, wl, wr, r_val, P, V = Our_SVD(A', B')

println("U size: ", size(U))
println("V_til size: ", size(V_til))
println("H size: ", size(H))
println("D1 size: ", size(D1))
println("D2_our size: ", size(D2_our))
println("D1_til: ", size(D1_til))
println("D2_til: ", size(D2_til))
println("bl: ", bl)
println("br: ", br)
println("wl: ", wl)
println("wr: ", wr)
println("r_val: ", r_val)
println("u_wires: ", bl+r_val+wl)
println("v_wires: ", wr+r_val+br)

Sorting

In [ ]:
U_new, V_new, D1_til_sorted, D2_til_sorted, D1_new, D2_our_new, H_new = sort_and_rebuild(
  U, V_til, D1, D2_our, D1_til, D2_til, H, bl, br, wl, wr, r_val
)

println("Dimensions:")
println("U_new: ", size(U_new))
println("V_new: ", size(V_new))
println("D1_til_sorted", size(D1_til_sorted))
println("D2_til_sorted", size(D2_til_sorted))
println("D1_new: ", size(D1_new))
println("D2_new: ", size(D2_our_new)) 
println("H_new: ", size(H_new))

### Error Calculation

In [ ]:
display(size(H[bl+1:bl+r_val,:]))
display(norm(I - (D1_new'*D1_new + D2_our_new'*D2_our_new)))
display(norm(I - (D1'*D1 + D2_our'*D2_our)))

In [ ]:
# Error from Julia's method
#println("\n A - U*D1*H: ", norm(A - U*D1*H))
#println("\n B - V_til*D2_our*H: ", norm(B - V_til*D2_our*H))


# Error after reordering
#println("\n A - U_new*D1_new*H_new: ", norm(A - U_new*D1_new*H_new)) 
#println("\n B - V_new*D2_our_new*H_new: ", norm(B - V_new*D2_our_new*H_new)) 

# Difference between Julia's method and reordering
println("\n A diff ", norm(U*D1*H - U_new*D1_new*H_new))
println("\n B diff ", norm(V_til*D2_our*H - V_new*D2_our_new*H_new))

### Angle Plot

- D1

In [ ]:
plot_angles(D1_new)

In [ ]:
plot_angles(D1_til_sorted)

- D2

In [ ]:
plot_angles(D2_our_new)

In [ ]:
plot_angles(D2_til_sorted)

### Find extremes (indices where the tails begin)

In [ ]:
function find_cutoff_indices(v, lim_sup, lim_inf)
    last_above = findlast(abs.(v) .> lim_sup)
    first_below = findfirst(x -> x < lim_inf, v)
    return (first_below, last_above)
end

function show_all(v)
    for i in 1:length(v)
        println("[$i] = ", v[i])
    end
end

In [ ]:
plot_vec(H[80,:])

In [ ]:
v = diag(D1_new)
first_index, last_index = find_cutoff_indices(v, 0, 1)

In [ ]:
plot_recon_A(D1_new, H_new', 511, media_A)

In [ ]:
plot_recon_A(D2_our_new, H_new', 511, media_B)

In [ ]:
v = diag(D1_new)
first_index, last_index = find_cutoff_indices(v, 0, 1)

println("First index with value below 1: ", first_index)
println("Last index with value above 0: ", last_index)

show_all(v)


### Plot of H

In [ ]:
plot_matrix(H_new', "H", nothing, nothing, "H_new.png", first_index, last_index)

In [ ]:
#plot_matrix(H_new, "H_new", bl+r_val+br, savepath="H.png")

### Reduction of H (removing the intersection)

- GIF

In [ ]:
using FileIO

function gif_matrix(
    M,
    titulo;
    media=nothing,
    savepath="scroll_imagens.gif",
    fps=nothing,
    colormap=:viridis,
    ini=nothing,
    fim=nothing
)
    n_total = size(M, 2)  # We now use the number of columns

    # Adjust start and end with default values
    ini = ini === nothing ? 1 : clamp(ini, 1, n_total)
    fim = fim === nothing ? n_total : clamp(fim, ini, n_total)

    # Submatrix for display (using columns)
    M_exibir = M[:, ini:fim]

    # Aplicar mean, se fornecida
    if media !== nothing
        media_vec = vec(media)  # ensure that it is a 784-vector (correct size)
        @assert length(media_vec) == size(M_exibir, 1) "media deve ter comprimento igual ao nº de linhas de M"
        M_exibir = M_exibir .+ media_vec
    end

    # Adjust the number of images to display
    k = size(M_exibir, 2) # now using the number of columns

    anim = @animate for i in 1:k
        img = reshape(M_exibir[:, i], (28, 28))  # We now take the i-th column
        heatmap(
            img[:, end:-1:1]',
            axis=nothing,
            colorbar=false,
            framestyle=:none,
            aspect_ratio=1,
            title="$titulo\nImagem $(ini + i - 1) de $fim",
            c=colormap
        )
    end

    gif(anim, savepath, fps=fps)
end


In [ ]:
#gif_matrix(H_new, "Componentes H", r_val, savepath="H_scroll.gif", fps=15, colormap=:viridis)

In [ ]:
gif_matrix(H_new', "Columns of H"; media=nothing, savepath="H.gif", fps=15, ini=first_index, fim=last_index)

### Image closest to 45 degrees (Principal Shared Component)

In [ ]:
angles = get_angles_D1(D2_our_new)
highlight_closest(angles, 45,1)

In [ ]:
plot_recon_A(D1_new, H_new', 211, media_A)

In [ ]:
plot_recon_A(D2_our_new, H_new', 110, media_B)

## Exploring toward the classifier

In [ ]:
D1_new_reduzido, D2_our_new_reduzido = set_to_0_closest_to_45(D1_new, D2_our_new,200)

In [ ]:
plot_angles(D1_new_reduzido)

In [ ]:
plot_angles(D2_our_new_reduzido)

### Classification with the Inverse

In [ ]:
function make_D1_D2_invertible(D1_til, D2_til, bl, br, wl, wr, r)
    D1_til_inv = copy(D1_til)
    D2_til_inv = copy(D2_til)

    for i in 1:size(D1_til, 1)
        D1_til_inv[i, i] = D1_til[i, i] != 0 ? 1 / D1_til[i, i] : 0
    end

    for i in 1:size(D2_til, 1)
        D2_til_inv[i, i] = D2_til[i, i] != 0 ? 1 / D2_til[i, i] : 0
    end

    # Construction of the inverted blocks
    D1_inv = [zeros(bl, bl)    zeros(bl, r)    zeros(bl, br);
              zeros(r, bl)    D1_til_inv       zeros(r, br);
              zeros(wl, bl)    zeros(wl, r)   zeros(wl, br)]

    D2_inv = [zeros(wr, bl)    zeros(wr, r)    zeros(wr, br);
              zeros(r, bl)    D2_til_inv       zeros(r, br);
              zeros(br, bl)    zeros(br, r)   zeros(br, br)]

    return D1_inv, D2_inv
end


In [ ]:
D1_inv, D2_inv = make_D1_D2_invertible(D1_til_sorted, D2_til_sorted, bl, br, wl, wr, r_val)

### Classification

generic classifier for any instance v

In [ ]:
c1 = vec(H_new' \ v1) # solve the least-squares problem: vector c contains the coefficients that best approximate vector v as a linear combination of the columns of matrix H_new'

wA = D1_new_reduzido * c1
wB = D2_our_new_reduzido* c1


classificador = norm(wA)/norm(wB)
println("Classifier (norm WA / norm WB): ", classificador)
if classificador > 1
    println("Classified as digit $digitA")
else
    println("Classified as digit $digitB")
end

energia_c = norm(c1)^2
energia_A = norm(wA)^2
energia_B = norm(wB)^2


soma_energias_componentes = energia_A + energia_B

diferenca = energia_c - soma_energias_componentes
println("\n||c||²:           ", energia_c)
println("||w_A||²:           ", energia_A)
println("||w_B||²:           ", energia_B)
println("Difference (Energy_c - Sum):     ", diferenca)

if isapprox(energia_c, soma_energias_componentes)
    println("\n The identity ||c||² ≈ ||w_A||² + ||w_B||² has been verified!")
else
    println("\n X")
end



In [ ]:
function classify(v, digitA, digitB, D1, D2, H)
    c = vec(H' \ v)

    wA = D1 * c
    wB = D2 * c

    classificador = norm(wA) / norm(wB)

    return classificador > 1.0 ? digitA : digitB
end

In [ ]:
function classify(v, digitA, digitB, D1, D2, H)
    # Solve H' * c = v
    c = vec(H' \ v)

    # Projections onto the cosine and sine bases
    wA = D1 * c
    wB = D2 * c

    # Compute the classifier
    classificador = norm(wB) / norm(wA)
    resultado = classificador > 1.0 ? digitB : digitA

    # Compute the angle in radians and convert it to degrees
    theta_rad = atan(norm(wB), norm(wA))
    theta_deg = theta_rad * (180 / π)

    return resultado, theta_deg
end

In [ ]:
v = get_MNIST_vectors(1,1)'
plot_vec(v)

In [ ]:
digit, theta = classify(v, digitA,digitB, D1_new, D2_our_new, H_new)

In [ ]:
#v = H_new'[:,253]

## Classifier with Theta(z)

In [ ]:
function classify_all_A(digit_A, D1, D2, H; threshold=1.0) 
    X = get_all_MNIST(digit_A, split=:test)    
 
    acertos = 0 
    total = size(X, 2) 
    angles = Float64[]  # Array to store the angles
    
    print("Classifying $total images of digit $digit_A...\n") 
 
    for i in 1:total 
        v = X[:, i] 
        c = vec(H' \ v)  # least squares 
        w1 = D1 * c 
        w2 = D2 * c 
        ratio = norm(w2) / norm(w1) 
        
        # Compute the angle in degrees
        theta_rad = atan(norm(w2), norm(w1))
        theta_deg = theta_rad * (180 / π)
        push!(angles, theta_deg)
 
        # decision rule with an indecision zone 
        predicao = ratio < threshold ? digit_A : -1  
        if predicao == digit_A 
            acertos += 1 
        end 
    end 
    
    # Compute angle statistics
    media_angles = mean(angles)
     deviation_angles = std(angles)
    accuracy = acertos / total
 
    return acertos, accuracy, angles, media_angles,  deviation_angles
end

function classify_all_B(digit_B, D1, D2, H; threshold=1.0) 
    X = get_all_MNIST(digit_B, split=:test)    
 
    acertos = 0 
    total = size(X, 2) 
    angles = Float64[]  # Array to store the angles
    
    print("Classifying $total images of digit $digit_B...\n") 
 
    for i in 1:total 
        v = X[:, i] 
        c = vec(H' \ v)  # least squares 
        w1 = D1 * c 
        w2 = D2 * c 
        ratio = norm(w2) / norm(w1) 
        
        # Compute the angle in degrees
        theta_rad = atan(norm(w2), norm(w1))
        theta_deg = theta_rad * (180 / π)
        push!(angles, theta_deg)
 
        # decision rule with an indecision zone 
        predicao = ratio > threshold ? digit_B : -1 
        if predicao == digit_B 
            acertos += 1 
        end 
    end 
    
    # Compute angle statistics
    media_angles = mean(angles)
     deviation_angles = std(angles)
    accuracy = acertos / total
 
    return acertos, accuracy, angles, media_angles,  deviation_angles
end

# Helper function to display the results in an organized way
function display_results(nome_digit, acertos, total, accuracy, media_angles,  deviation_angles)
    println("\n=== Results for digit $nome_digit ===")
    println("Correct: $acertos out of $total")
    println("Accuracy: $(round(accuracy * 100, digits=2))%")
    println("Mean of the angles: $(round(media_angles, digits=2))°")
    println("Standard deviation of the angles: $(round( deviation_angles, digits=2))°")
end

# Example of use:
function test_classification(digit_A, digit_B, D1, D2, H; threshold=1.0)
    println("Testing classification between digits $digit_A and $digit_B")
    
    # Test set A
    acertos_A, accuracy_A, angles_A, media_A,  deviation_A = classify_all_A(digit_A, D1, D2, H; threshold=threshold)
    total_A = length(angles_A)
    display_results(digit_A, acertos_A, total_A, accuracy_A, media_A,  deviation_A)
    
    # Test set B  
    acertos_B, accuracy_B, angles_B, media_B,  deviation_B = classify_all_B(digit_B, D1, D2, H; threshold=threshold)
    total_B = length(angles_B)
    display_results(digit_B, acertos_B, total_B, accuracy_B, media_B,  deviation_B)
    
    # Accuracy geral
    accuracy_geral = (acertos_A + acertos_B) / (total_A + total_B)
    println("\n=== Overall Summary ===")
    println("Overall accuracy: $(round(accuracy_geral * 100, digits=2))%")
    
    return (angles_A, angles_B, accuracy_geral)
end


In [ ]:
function plot_angle_histogram(angles_A, angles_B, digit_A, digit_B; 
                                  bins=30, alpha=0.6, figsize=(1200, 500))
    
    # Compute statistics
    media_A = mean(angles_A)
    media_B = mean(angles_B)
     deviation_A = std(angles_A)
     deviation_B = std(angles_B)
    
    # Create the histogram with both sets - flattened format for the paper
    p = histogram(angles_A, 
                 bins=bins, 
                 alpha=alpha, 
                 label="Digit $digit_A (σ=$(round( deviation_A, digits=1))°)", 
                 color=:steelblue,
                 xlabel="Angle θ (degrees)",
                 ylabel="Frequency",
                 title="",  # No title - it will be the caption in LaTeX
                 size=figsize,
                 # Increase font sizes
                 guidefontsize=24,
                 tickfontsize=18,
                 legendfontsize=16,
                 left_margin=12Plots.mm,
                 bottom_margin=12Plots.mm,
                 right_margin=4Plots.mm,
                 top_margin=4Plots.mm,
                 dpi=300,  # High resolution for the paper
                 framestyle=:box)
    
    # Add the second histogram
    histogram!(p, angles_B, 
              bins=bins, 
              alpha=alpha, 
              label="Digit $digit_B (σ=$(round( deviation_B, digits=1))°)", 
              color=:coral)
    
    # Add vertical lines for the means - thicker and more visible
    vline!([media_A], 
           color=:steelblue, 
           linestyle=:dash, 
           linewidth=4,
           label="Mean $digit_A: $(round(media_A, digits=1))°")
    
    vline!([media_B], 
           color=:coral, 
           linestyle=:dash, 
           linewidth=4,
           label="Mean $digit_B: $(round(media_B, digits=1))°")
    
    # Position the legend
    plot!(legend=:topright, 
          legendfontsize=14,
          legend_background_color=:white,
          legend_foreground_color=:black)
    
    # Save as PNG
    filename = ".\\data\\comparacoes_digits\\distribution_of_theta_$(digit_A)_$(digit_B).png"
    savefig(p, filename)
    println("Image saved at: $filename")
    
    return p
end





In [ ]:
#angles_A, angles_B, _ = test_classification(digitA, digitB, D1_new, D2_our_new, H_new;  threshold=1.0)

In [ ]:
#p = plot_angle_histogram(angles_A, angles_B, 0, 7)
#display(p)

### Visualization pipeline for extremes and the PSC

In [ ]:
function visualize_H_optimization(digitA, digitB; radius=3)
    println("[visualize_H_optimization] Preparing data for digits $digitA and $digitB...")
    A_prep, B_prep, media_A_prep, media_B_prep, U_prep, V_prep, D1_prep, D2_our_prep, H_prep = prepare_data(digitA, digitB)
    println("[visualize_H_optimization] Separating diagonal components...")
    d = diag(D1_prep) # Could also be taken from D2_our_prep
    println("[visualize_H_optimization] Finding extreme values...")
    shared_indices = find_shared_indices(d)
    println("[visualize_H_optimization] Plotting extreme vectors of A...")
    plots_extreme_A = [plot_vec(H_prep[shared_indices[1] + i, :], display=false) for i in 0:(radius-1)]
    combined_plot_extremes_A = plot(plots_extreme_A..., layout=(1, radius), size=(300*radius,300))
    display(combined_plot_extremes_A)
    println("Extreme indices of A: ", shared_indices[1:radius])
    println("[visualize_H_optimization] Plotting extreme vectors of B...")
    plots_extreme_B = [plot_vec(H_prep[shared_indices[end] - i, :], display=false) for i in 0:(radius-1)]
    combined_plot_extremes_B = plot(plots_extreme_B..., layout=(1, radius), size=(300*radius,300))
    display(combined_plot_extremes_B)
    println("Extreme indices of B: ", shared_indices[end:-1:end-radius+1])
    println("[visualize_H_optimization] Plotting candidate PSC vectors...")
    psc_candidates = highlight_closest(d, sqrt(2)/2,radius)
    plots_psc_candidates = [plot_vec(H_prep[i, :], display=false) for i in psc_candidates]
    combined_plot_psc = plot(plots_psc_candidates..., layout=(1, radius), size=(300*radius,300))
    display(combined_plot_psc)
    println("Candidate PSC indices: ", psc_candidates)

    return H_prep
end

function find_shared_indices(d, lower_lim = 0.0, upper_lim = 1.0)
    shared_indices = []
    for i in 1:length(d)
        if d[i] > lower_lim && d[i] < upper_lim
            push!(shared_indices, i)
        end
    end
    return shared_indices
end

function save_H_index_images(H, indices)
    for idx in indices
        img_vec = H[idx, :]
        img = reshape(img_vec, (28, 28))
        heatmap(
            img[:, end:-1:1]',
            axis=nothing,
            colorbar=false,
            framestyle=:none,
            aspect_ratio=1,
            title="H component index $idx",
            c=:viridis
        )
        savefig(".\\data\\problema_otimizacao\\H_component_$idx.png")
    end
end

In [ ]:
H_to_optimization_images = visualize_H_optimization(4, 9, radius=5)

In [ ]:
save_H_index_images(H_to_optimization_images, [81, 509, 314]) 

### More structured test to plot histograms for several pairs

In [ ]:
function prepare_data(digitA, digitB)
    println("[prepare_data] Starting data preparation...")
    n = 900
    m = 800
    println("[prepare_data] Creating matrices...")
    A_local, B_local = create_MNIST_matrices(digitA, digitB, n, m)
    println("[prepare_data] Matrices created.")
    
    println("[prepare_data] Centering matrices...")
    media_A_local = mean(A_local, dims=2) |> vec  # dims=2 for column-wise mean
    media_B_local = mean(B_local, dims=2) |> vec  # dims=2 for column-wise mean
    display(size(media_A_local))
    A_centered = A_local .- media_A_local
    B_centered = B_local .- media_B_local
    println("[prepare_data] Matrices centered.")
    
    # GSVD decomposition
    println("[prepare_data] Running Our_SVD...")
    U_svd, V_til_svd, H_svd, D1_til_svd, D2_til_svd, D1_svd, D2_our_svd, bl_svd, br_svd, wl_svd, wr_svd, r_val_svd, P_svd, V_svd_orig = Our_SVD(A_centered', B_centered')
    println("[prepare_data] Our_SVD finished.")

    println("[prepare_data] Running sort_and_rebuild...")
    U_sorted, V_sorted, D1_til_sorted, D2_til_sorted, D1_sorted, D2_our_sorted, H_sorted = sort_and_rebuild(
        U_svd, V_til_svd, D1_svd, D2_our_svd, D1_til_svd, D2_til_svd, H_svd, bl_svd, br_svd, wl_svd, wr_svd, r_val_svd
    )
    println("[prepare_data] sort_and_rebuild finished.")
    
    return A_centered, B_centered, media_A_local, media_B_local, U_sorted, V_sorted, D1_sorted, D2_our_sorted, H_sorted
end

# Global variable to store all histograms
all_histograms = []
all_angles_data = []

function classify_more(digitA, digitB)
    println("[classify_more] Preparing data for digits $digitA and $digitB...")
    A_prep, B_prep, media_A_prep, media_B_prep, U_prep, V_prep, D1_prep, D2_our_prep, H_prep = prepare_data(digitA, digitB)
    println("[classify_more] Data prepared. Classifying both digits...")
    angles_A, angles_B, accuracy_geral = test_classification(digitA, digitB, D1_prep, D2_our_prep, H_prep; threshold=1.0)
    println("[classify_more] Classification completed. Plotting histogram...")
    p = plot_angle_histogram(angles_A, angles_B, digitA, digitB)
    println("[classify_more] Histogram plotted. Saving results...")
    push!(all_histograms, p)
    push!(all_angles_data, (digitA=digitA, digitB=digitB, angles_A=angles_A, angles_B=angles_B))
    println("[classify_more] Results saved.")
    return angles_A, angles_B, accuracy_geral, p
end

function plot_all_histograms()
    println("[plot_all_histograms] Plotting all histograms...")
    n_plots = length(all_histograms)
    if n_plots == 0
        println("No histogram to plot.")
        return
    end
    ncols = min(3, n_plots)
    nrows = ceil(Int, n_plots / ncols)
    combined_plot = plot(all_histograms..., layout=(nrows, ncols), size=(800*ncols, 600*nrows))
    display(combined_plot)
    println("[plot_all_histograms] Plots displayed.")
    return combined_plot
end

function save_all_angles_to_csv(filename="angles_results.csv")
    println("[save_all_angles_to_csv] Saving angles to CSV...")
    if isempty(all_angles_data)
        println("No data to save.")
        return
    end
    rows = []
    for data in all_angles_data
        for ang in data.angles_A
            push!(rows, (digitA=data.digitA, digitB=data.digitB, digit_class=data.digitA, angle=ang))
        end
        for ang in data.angles_B
            push!(rows, (digitA=data.digitA, digitB=data.digitB, digit_class=data.digitB, angle=ang))
        end
    end
    df = DataFrame(rows)
    CSV.write(filename, df)
    println("[save_all_angles_to_csv] Data saved to $filename")
    return df
end


In [ ]:
println("\n>>> Pair: Digits 1 e 5")
classify_more(1, 5)


println("\n>>> Pair: Digits 3 e 9")
classify_more(3, 9)

println("\n>>> Pair: Digits 4 e 9")
classify_more(4, 9)

println("\n>>> Pair: Digits 0 e 7")
classify_more(0, 7)


println("\n=== All classifications completed ===")

# Plot all histograms together
plot_all_histograms()

# Save all angles to CSV
save_all_angles_to_csv("angles_results_all_pairs.csv")

### ### Angle histograms to inspect the data distribution

Idea: look at the theta distributions independently of the classes (considering all samples from the same class together)

In [ ]:
function plot_posterior_angle_distribution(
    angles_A, angles_B, digit_A, digit_B;
    bins=30, figsize=(1200, 500)
)

    # =========================
    # Shared bins
    # =========================
    angles = vcat(angles_A, angles_B)
    edges = range(0, 90, length=bins+1)
    centers = (edges[1:end-1] .+ edges[2:end]) ./ 2
    width = edges[2] - edges[1]

    # =========================
    # Counts per class
    # =========================
    counts_A = zeros(Float64, bins)
    counts_B = zeros(Float64, bins)

    for θ in angles_A
        i = clamp(searchsortedlast(edges, θ), 1, bins)
        counts_A[i] += 1
    end

    for θ in angles_B
        i = clamp(searchsortedlast(edges, θ), 1, bins)
        counts_B[i] += 1
    end

    # =========================
    # Per-bin normalization
    # =========================
    total = counts_A .+ counts_B
    empty_bins = total .== 0
    total[empty_bins] .= 1.0   # avoid division by zero

    post_A = counts_A ./ total
    post_B = counts_B ./ total

    # =========================
    # Background for empty bins
    # =========================
    p = plot(
        xlabel = "Angle θ (degrees)",
        ylabel = "Posterior probability",
        ylim = (0, 1),
        xlim = (0, 90),
        size = figsize,
        xticks=0:15:90,
        guidefontsize = 24,
        tickfontsize = 18,
        legendfontsize = 16,
        left_margin = 12Plots.mm,
        bottom_margin = 12Plots.mm,
        right_margin = 4Plots.mm,
        top_margin = 4Plots.mm,
        dpi = 300,
        framestyle = :box,
        legend = :topright,
        legend_background_color = :white,
        legend_foreground_color = :black
    )

    # Add light blue/orange bars for empty bins
    for i in 1:bins
        if empty_bins[i]
            color = centers[i] < 45 ? :deepskyblue : :lightsalmon
            bar!(
                [centers[i]],
                [1.0],
                bar_width = width,
                color = color,
                alpha = 0.3,
                label = ""
            )
        end
    end

    # =========================
    # Stacked posterior histogram
    # =========================
    bar!(
        centers,
        post_A .+ post_B,  # Bottom layer (full height)
        bar_width = width,
        label = "Digit $digit_B",
        color = :coral
    )
    
    # Add the bottom layer on top
    bar!(
        centers,
        post_A,
        bar_width = width,
        label = "Digit $digit_A",
        color = :steelblue
    )

    # =========================
    # Save
    # =========================
    filename = ".\\data\\comparacoes_digits\\posteriori_theta_normalized_$(digit_A)_$(digit_B).png"
    savefig(p, filename)
    println("Image saved at: $filename")

    return p
end

In [ ]:
# Global variable to store all histograms
all_posteriori = []
all_posteriori_angles_data = []

function posterior_theta_graph(digitA, digitB)
    println("[posterior_theta_graph] Preparing data for digits $digitA and $digitB...")
    A_prep, B_prep, media_A_prep, media_B_prep, U_prep, V_prep, D1_prep, D2_our_prep, H_prep = prepare_data(digitA, digitB)
    println("[posterior_theta_graph] Data prepared. Classifying both digits...")
    angles_A, angles_B, accuracy_geral = test_classification(digitA, digitB, D1_prep, D2_our_prep, H_prep; threshold=1.0)
    println("[posterior_theta_graph] Classification completed. Plotting histogram...")
    p = plot_posterior_angle_distribution(angles_A, angles_B, digitA, digitB)
    println("[posterior_theta_graph] Histogram plotted. Saving results...")
    push!(all_posteriori, p)
    push!(all_posteriori_angles_data, (digitA=digitA, digitB=digitB, angles_A=angles_A, angles_B=angles_B))
    println("[posterior_theta_graph] Results saved.")
    return angles_A, angles_B, accuracy_geral, p
end


function plot_all_posteriori()
    println("[plot_all_histograms] Plotting all histograms...")
    n_plots = length(all_posteriori)
    if n_plots == 0
        println("No histogram to plot.")
        return
    end
    ncols = min(3, n_plots)
    nrows = ceil(Int, n_plots / ncols)
    combined_plot = plot(all_posteriori..., layout=(nrows, ncols), size=(800*ncols, 600*nrows))
    display(combined_plot)
    println("[plot_all_histograms] Plots displayed.")
    return combined_plot
end



In [ ]:
posterior_theta_graph(1, 5)

In [ ]:
posterior_theta_graph(1, 5)
posterior_theta_graph(0, 7)
posterior_theta_graph(4, 9)
posterior_theta_graph(3, 9)


plot_all_posteriori()


### ### Angle histograms to inspect the data distribution

Idea: look at the theta distributions independently of the classes (considering all samples from the same class together)

In [ ]:
function plot_posterior_angle_distribution(
    angles_A, angles_B, digit_A, digit_B;
    bins=30, figsize=(1200, 500)
)

    # =========================
    # Shared bins
    # =========================
    angles = vcat(angles_A, angles_B)
    edges = range(0, 90, length=bins+1)
    centers = (edges[1:end-1] .+ edges[2:end]) ./ 2
    width = edges[2] - edges[1]

    # =========================
    # Counts per class
    # =========================
    counts_A = zeros(Float64, bins)
    counts_B = zeros(Float64, bins)

    for θ in angles_A
        i = clamp(searchsortedlast(edges, θ), 1, bins)
        counts_A[i] += 1
    end

    for θ in angles_B
        i = clamp(searchsortedlast(edges, θ), 1, bins)
        counts_B[i] += 1
    end

    # =========================
    # Per-bin normalization
    # =========================
    total = counts_A .+ counts_B
    empty_bins = total .== 0
    total[empty_bins] .= 1.0   # avoid division by zero

    post_A = counts_A ./ total
    post_B = counts_B ./ total

    # =========================
    # Background for empty bins
    # =========================
    p = plot(
        xlabel = "Angle θ (degrees)",
        ylabel = "Posterior probability",
        ylim = (0, 1),
        xlim = (0, 90),
        size = figsize,
        xticks=0:15:90,
        guidefontsize = 24,
        tickfontsize = 18,
        legendfontsize = 16,
        left_margin = 12Plots.mm,
        bottom_margin = 12Plots.mm,
        right_margin = 4Plots.mm,
        top_margin = 4Plots.mm,
        dpi = 300,
        framestyle = :box,
        legend = :topright,
        legend_background_color = :white,
        legend_foreground_color = :black
    )

    # Add light blue/orange bars for empty bins
    for i in 1:bins
        if empty_bins[i]
            color = centers[i] < 45 ? :deepskyblue : :lightsalmon
            bar!(
                [centers[i]],
                [1.0],
                bar_width = width,
                color = color,
                alpha = 0.3,
                label = ""
            )
        end
    end

    # =========================
    # Stacked posterior histogram
    # =========================
    bar!(
        centers,
        post_A .+ post_B,  # Bottom layer (full height)
        bar_width = width,
        label = "Digit $digit_B",
        color = :coral
    )
    
    # Add the bottom layer on top
    bar!(
        centers,
        post_A,
        bar_width = width,
        label = "Digit $digit_A",
        color = :steelblue
    )

    # =========================
    # Save
    # =========================
    filename = ".\\data\\comparacoes_digits\\posteriori_theta_normalized_$(digit_A)_$(digit_B).png"
    savefig(p, filename)
    println("Image saved at: $filename")

    return p
end

In [ ]:
# Global variable to store all histograms
all_posteriori = []
all_posteriori_angles_data = []

function posterior_theta_graph(digitA, digitB)
    println("[posterior_theta_graph] Preparing data for digits $digitA and $digitB...")
    A_prep, B_prep, media_A_prep, media_B_prep, U_prep, V_prep, D1_prep, D2_our_prep, H_prep = prepare_data(digitA, digitB)
    println("[posterior_theta_graph] Data prepared. Classifying both digits...")
    angles_A, angles_B, accuracy_geral = test_classification(digitA, digitB, D1_prep, D2_our_prep, H_prep; threshold=1.0)
    println("[posterior_theta_graph] Classification completed. Plotting histogram...")
    p = plot_posterior_angle_distribution(angles_A, angles_B, digitA, digitB)
    println("[posterior_theta_graph] Histogram plotted. Saving results...")
    push!(all_posteriori, p)
    push!(all_posteriori_angles_data, (digitA=digitA, digitB=digitB, angles_A=angles_A, angles_B=angles_B))
    println("[posterior_theta_graph] Results saved.")
    return angles_A, angles_B, accuracy_geral, p
end


function plot_all_posteriori()
    println("[plot_all_histograms] Plotting all histograms...")
    n_plots = length(all_posteriori)
    if n_plots == 0
        println("No histogram to plot.")
        return
    end
    ncols = min(3, n_plots)
    nrows = ceil(Int, n_plots / ncols)
    combined_plot = plot(all_posteriori..., layout=(nrows, ncols), size=(800*ncols, 600*nrows))
    display(combined_plot)
    println("[plot_all_histograms] Plots displayed.")
    return combined_plot
end



In [ ]:
posterior_theta_graph(1, 5)

In [ ]:
posterior_theta_graph(1, 5)
posterior_theta_graph(0, 7)
posterior_theta_graph(4, 9)
posterior_theta_graph(3, 9)


plot_all_posteriori()


## Classifier using Fashion-MNIST

0 - T-shirt/top
1 - Trouser (Pants)
2 - Pullover
3 - Dress
4 - Coat
5 - Sandal
6 - Shirt
7 - Sneaker
8 - Bag
9 - Ankle boot

### Adapted functions

In [ ]:
function get_all_Fashion_MNIST(clothing class; split=:test)
    imgs, labels = FashionMNIST(split=split)[:]

    imgs = Float64.(imgs) ./ 255

    idx = findall(labels .== clothing class)

    X = hcat([vec(imgs[:, :, i]) for i in idx]...) # place the images as columns
    return X
end


function create_Fashion_MNIST_matrices(clothing classA, clothing classB, n, m)

    imgs, labels = FashionMNIST(split=:train)[:]

     # normalize
    imgs = Float64.(imgs) ./ 255 

    # find the clothing classes
    idxA = findall(labels .== clothing classA)
    idxB = findall(labels .== clothing classB)

    # handle errors
    if length(idxA) < n || length(idxB) < m
        error("There are not enough images for the chosen clothing classes.")
    end

    # shuffle the indices
    Random.seed!(1234)  
    idxA = Random.shuffle(idxA)[1:n]
    idxB = Random.shuffle(idxB)[1:m]

     # Vectorize and stack as columns
    A = hcat([vec(imgs[:, :, i]) for i in idxA]...)
    B = hcat([vec(imgs[:, :, i]) for i in idxB]...)
    

    return A, B
end

function plot_fashion_vector(v)
    img = reshape(v, (28, 28))
    heatmap(
        img[:, end:-1:1]',
        axis=nothing,
        colorbar=false,
        framestyle=:none,
        aspect_ratio=1,
        title="Fashion-MNIST image",
        c=:viridis
    )
end

In [ ]:
function plot_fashion_angle_histogram(angles_A, angles_B, clothing class_A, clothing class_B; 
                                  bins=30, alpha=0.6, figsize=(1200, 500))
    
    # Mapping of indices to clothing class names
    clothing class_names = Dict(
        0 => "T-shirt/top",
        1 => "Trouser",
        2 => "Pullover",
        3 => "Dress",
        4 => "Coat",
        5 => "Sandal",
        6 => "Shirt",
        7 => "Sneaker",
        8 => "Bag",
        9 => "Ankle boot"
    )
    
    # Get the clothing class names
    nome_A = get(clothing class_names, clothing class_A, "Clothing class $clothing class_A")
    nome_B = get(clothing class_names, clothing class_B, "Clothing class $clothing class_B")
    
    # Compute statistics
    media_A = mean(angles_A)
    media_B = mean(angles_B)
     deviation_A = std(angles_A)
     deviation_B = std(angles_B)
    
    # Create the histogram with both sets - flattened format for the paper
    p = histogram(angles_A, 
                 bins=bins, 
                 alpha=alpha, 
                 label="$nome_A (σ=$(round( deviation_A, digits=1))°)", 
                 color=:steelblue,
                 xlabel="Angle θ (degrees)",
                 xlims=(0,90),
                 ylabel="Frequency",
                 ylims=(0,350),
                 title="",  # No title - it will be the caption in LaTeX
                 size=figsize,
                 # Increase font sizes
                 guidefontsize=24,
                 tickfontsize=18,
                 legendfontsize=16,
                 left_margin=12Plots.mm,
                 bottom_margin=12Plots.mm,
                 right_margin=4Plots.mm,
                 top_margin=4Plots.mm,
                 dpi=300,  # High resolution for the paper
                 framestyle=:box)
    
    # Add the second histogram
    histogram!(p, angles_B, 
              bins=bins, 
              alpha=alpha, 
              label="$nome_B (σ=$(round( deviation_B, digits=1))°)", 
              color=:coral)
    
    # Add vertical lines for the means - thicker and more visible
    vline!([media_A], 
           color=:steelblue, 
           linestyle=:dash, 
           linewidth=4,
           label="Mean $nome_A: $(round(media_A, digits=1))°")
    
    vline!([media_B], 
           color=:coral, 
           linestyle=:dash, 
           linewidth=4,
           label="Mean $nome_B: $(round(media_B, digits=1))°")
    
    # Position the legend
    plot!(legend=:topright, 
          legendfontsize=14,
          legend_background_color=:white,
          legend_foreground_color=:black)
    
    # Save as PNG
    filename = ".\\data\\comparacoes_fashion\\distribution_of_theta_Fashion_$(clothing class_A)_$(clothing class_B).png"
    savefig(p, filename)
    println("Image saved at: $filename")
    
    return p
end

### Initial tests

In [ ]:
n = 900
m = 800

clothing classA = 0  # T-shirt/top
clothing classB = 7  # Sneaker
A_fashion, B_fashion = create_Fashion_MNIST_matrices(clothing classA, clothing classB, n, m)
println("Size of matrix A_fashion: ", size(A_fashion))  
println("Size of matrix B_fashion: ", size(B_fashion))  
plot_matrix(A_fashion, "A", 20)
plot_matrix(B_fashion, "B",20)
# centering
media_A_fashion = mean(A_fashion, dims=2) |> vec  # dims
media_B_fashion = mean(B_fashion, dims=2) |> vec # dims
A_fashion = A_fashion .- media_A_fashion
B_fashion = B_fashion .- media_B_fashion
plot_matrix(A_fashion, "A", 20)
plot_matrix(B_fashion, "B", 20)


In [ ]:
U_fashion, V_til_fashion, H_fashion, D1_til_fashion, D2_til_fashion, D1_fashion, D2_our_fashion, bl_fashion, br_fashion, wl_fashion, wr_fashion, r_val_fashion, P_fashion, V_fashion = Our_SVD(A_fashion', B_fashion')

U_fashion_new, V_fashion_new, D1_til_fashion_sorted, D2_til_fashion_sorted, D1_fashion_new, D2_our_fashion_new, H_fashion_new = sort_and_rebuild(
  U_fashion, V_til_fashion, D1_fashion, D2_our_fashion, D1_til_fashion, D2_til_fashion, H_fashion, bl_fashion, br_fashion, wl_fashion, wr_fashion, r_val_fashion
)

In [ ]:
first_index_fashion, last_index_fashion = find_cutoff_indices(diag(D1_fashion_new), 0, 7)
gif_matrix(H_fashion_new', "Columns of H Fashion-MNIST"; media=nothing, savepath="H_fashion.gif", fps=30, ini=first_index_fashion, fim=last_index_fashion)

In [ ]:
plot_matrix(H_fashion_new', "H", nothing, nothing, "H_new.png", first_index_fashion, last_index_fashion)

### Test several fashion pairs

In [ ]:
function test_Fashion_classification(clothing class_A, clothing class_B, D1, D2, H; threshold=1.0)
    println("Testing classification between clothing classes $clothing class_A and $clothing class_B")
    
    # Test set A
    acertos_A, accuracy_A, angles_A, media_A,  deviation_A = classify_all_A_Fashion(clothing class_A, D1, D2, H; threshold=threshold)
    total_A = length(angles_A)
    display_results(clothing class_A, acertos_A, total_A, accuracy_A, media_A,  deviation_A)
    
    # Test set B  
    acertos_B, accuracy_B, angles_B, media_B,  deviation_B = classify_all_B_Fashion(clothing class_B, D1, D2, H; threshold=threshold)
    total_B = length(angles_B)
    display_results(clothing class_B, acertos_B, total_B, accuracy_B, media_B,  deviation_B)
    
    # Accuracy geral
    accuracy_geral = (acertos_A + acertos_B) / (total_A + total_B)
    println("\n=== Overall Summary ===")
    println("Overall accuracy: $(round(accuracy_geral * 100, digits=2))%")
    
    return (angles_A, angles_B, accuracy_geral)
end

function classify_all_A_Fashion(clothing class_A, D1, D2, H; threshold=1.0) 
    X = get_all_Fashion_MNIST(clothing class_A, split=:test)    
 
    acertos = 0 
    total = size(X, 2) 
    angles = Float64[]  # Array to store the angles
    
    print("Classifying $total images of clothing class $clothing class_A...\n") 
 
    for i in 1:total 
        v = X[:, i] 
        c = vec(H' \ v)  # least squares 
        w1 = D1 * c 
        w2 = D2 * c 
        ratio = norm(w2) / norm(w1) 
        
        # Compute the angle in degrees
        theta_rad = atan(norm(w2), norm(w1))
        theta_deg = theta_rad * (180 / π)
        push!(angles, theta_deg)
 
        # decision rule with an indecision zone 
        predicao = ratio < threshold ? clothing class_A : -1  
        if predicao == clothing class_A 
            acertos += 1 
        end 
    end 
    
    # Compute angle statistics
    media_angles = mean(angles)
     deviation_angles = std(angles)
    accuracy = acertos / total
 
    return acertos, accuracy, angles, media_angles,  deviation_angles
end

function classify_all_B_Fashion(clothing class_B, D1, D2, H; threshold=1.0) 
    X = get_all_Fashion_MNIST(clothing class_B, split=:test)    
 
    acertos = 0 
    total = size(X, 2) 
    angles = Float64[]  # Array to store the angles
    
    print("Classifying $total images of clothing class $clothing class_B...\n") 
 
    for i in 1:total 
        v = X[:, i] 
        c = vec(H' \ v)  # least squares 
        w1 = D1 * c 
        w2 = D2 * c 
        ratio = norm(w2) / norm(w1) 
        
        # Compute the angle in degrees
        theta_rad = atan(norm(w2), norm(w1))
        theta_deg = theta_rad * (180 / π)
        push!(angles, theta_deg)
 
        # decision rule with an indecision zone 
        predicao = ratio > threshold ? clothing class_B : -1 
        if predicao == clothing class_B 
            acertos += 1 
        end 
    end 
    
    # Compute angle statistics
    media_angles = mean(angles)
     deviation_angles = std(angles)
    accuracy = acertos / total
 
    return acertos, accuracy, angles, media_angles,  deviation_angles
end

function prepare_Fashion_data(clothing classA, clothing classB)
    println("[prepare_data] Starting data preparation...")
    n = 900
    m = 800
    println("[prepare_data] Creating matrices...")
    A_local, B_local = create_Fashion_MNIST_matrices(clothing classA, clothing classB, n, m)
    println("[prepare_data] Matrices created.")
    
    println("[prepare_data] Centering matrices...")
    media_A_local = mean(A_local, dims=2) |> vec  # dims=2 for column-wise mean
    media_B_local = mean(B_local, dims=2) |> vec  # dims=2 for column-wise mean
    display(size(media_A_local))
    A_centered = A_local .- media_A_local
    B_centered = B_local .- media_B_local
    println("[prepare_data] Matrices centered.")
    
    # GSVD decomposition
    println("[prepare_data] Running Our_SVD...")
    U_svd, V_til_svd, H_svd, D1_til_svd, D2_til_svd, D1_svd, D2_our_svd, bl_svd, br_svd, wl_svd, wr_svd, r_val_svd, P_svd, V_svd_orig = Our_SVD(A_centered', B_centered')
    println("[prepare_data] Our_SVD finished.")

    println("[prepare_data] Running sort_and_rebuild...")
    U_sorted, V_sorted, D1_til_sorted, D2_til_sorted, D1_sorted, D2_our_sorted, H_sorted = sort_and_rebuild(
        U_svd, V_til_svd, D1_svd, D2_our_svd, D1_til_svd, D2_til_svd, H_svd, bl_svd, br_svd, wl_svd, wr_svd, r_val_svd
    )
    println("[prepare_data] sort_and_rebuild finished.")
    
    return A_centered, B_centered, media_A_local, media_B_local, U_sorted, V_sorted, D1_sorted, D2_our_sorted, H_sorted
end

# Global variable to store all histograms
all_histograms = []
all_angles_data = []

function classify_more_Fashion(clothing classA, clothing classB)
    println("[classify_more] Preparing data for clothing classes $clothing classA and $clothing classB...")
    A_prep, B_prep, media_A_prep, media_B_prep, U_prep, V_prep, D1_prep, D2_our_prep, H_prep = prepare_Fashion_data(clothing classA, clothing classB)
    println("[classify_more] Data prepared. Classifying both clothing classes...")
    angles_A, angles_B, accuracy_geral = test_Fashion_classification(clothing classA, clothing classB, D1_prep, D2_our_prep, H_prep; threshold=1.0)
    println("[classify_more] Classification completed. Plotting histogram...")
    p = plot_fashion_angle_histogram(angles_A, angles_B, clothing classA, clothing classB)
    println("[classify_more] Histogram plotted. Saving results...")
    push!(all_histograms, p)
    push!(all_angles_data, (clothing classA=clothing classA, clothing classB=clothing classB, angles_A=angles_A, angles_B=angles_B))
    println("[classify_more] Results saved.")
    return angles_A, angles_B, accuracy_geral, p
end

function plot_all_histograms()
    println("[plot_all_histograms] Plotting all histograms...")
    n_plots = length(all_histograms)
    if n_plots == 0
        println("No histogram to plot.")
        return
    end
    ncols = min(3, n_plots)
    nrows = ceil(Int, n_plots / ncols)
    combined_plot = plot(all_histograms..., layout=(nrows, ncols), size=(800*ncols, 600*nrows))
    display(combined_plot)
    println("[plot_all_histograms] Plots displayed.")
    return combined_plot
end

function save_all_Fashion_angles_to_csv(filename="angles_results_fashion.csv")
    println("[save_all_angles_to_csv] Saving angles to CSV...")
    if isempty(all_angles_data)
        println("No data to save.")
        return
    end
    rows = []
    for data in all_angles_data
        for ang in data.angles_A
            push!(rows, (clothing classA=data.clothing classA, clothing classB=data.clothing classB, clothing class_class=data.clothing classA, angle=ang))
        end
        for ang in data.angles_B
            push!(rows, (clothing classA=data.clothing classA, clothing classB=data.clothing classB, clothing class_class=data.clothing classB, angle=ang))
        end
    end
    df = DataFrame(rows)
    CSV.write(filename, df)
    println("[save_all_angles_to_csv] Data saved to $filename")
    return df
end

In [ ]:
angles_A, angles_B, accuracy_geral = test_Fashion_classification(0, 7, D1_fashion_new, D2_our_fashion_new, H_fashion_new; threshold=1.0)

In [ ]:
plot_fashion_angle_histogram(angles_A, angles_B, 0, 7)

In [ ]:
# t-shirt x coat 
println("\n>>> Pair: Clothing classes 0 (T-shirt/top) e 4 (Coat)")
classify_more_Fashion(0, 4)

# pullover x dress
println("\n>>> Pair: Clothing classes 2 (Pullover) e 3 (Dress)")
classify_more_Fashion(2, 3) 
# sneaker x ankle boot
println("\n>>> Pair: Clothing classes 7 (Sneaker) e 9 (Ankle boot)")
classify_more_Fashion(7, 9)

# t-shirt x sneaker
println("\n>>> Pair: Clothing classes 0 (T-shirt/top) e 7 (Sneaker)")
classify_more_Fashion(0, 7)

println("\n=== All classifications completed ===")

# Plot all histograms together
plot_all_histograms()
# Save all angles to CSV
save_all_Fashion_angles_to_csv("angles_results_all_pairs_fashion.csv")

#### classify using the matrix itself + different k values

In [ ]:
function classify_all_A_matrix(A, digit_A, D1, D2, H; threshold=1.0) 
    # A must be a matrix where each row is a vector to be classified
    # Transpose A so that each column is a vector
    X = A
    
    acertos = 0 
    total = size(X, 2) 
    angles = Float64[]  # Array to store the angles
    
    print("Classifying $total vectors from matrix A as digit $digit_A...\n") 
 
    for i in 1:total 
        v = X[:, i] 
        c = vec(H' \ v)  # least squares 
        w1 = D1 * c 
        w2 = D2 * c 
        ratio = norm(w2) / norm(w1) 
        
        # Compute the angle in degrees
        theta_rad = atan(norm(w2), norm(w1))
        theta_deg = theta_rad * (180 / π)
        push!(angles, theta_deg)
 
        # decision rule with an indecision zone 
        predicao = ratio < threshold ? digit_A : -1  
        if predicao == digit_A 
            acertos += 1 
        end 
    end 
    
    # Compute angle statistics
    media_angles = mean(angles)
     deviation_angles = std(angles)
    accuracy = acertos / total
 
    return acertos, accuracy, angles, media_angles,  deviation_angles
end

function classify_all_B_matrix(B, digit_B, D1, D2, H; threshold=1.0) 
    # B must be a matrix where each row is a vector to be classified
    # Transpose B so that each column is a vector
    X = B
    
    acertos = 0 
    total = size(X, 2) 
    angles = Float64[]  # Array to store the angles
    
    print("Classifying $total vectors from matrix B as digit $digit_B...\n") 
 
    for i in 1:total 
        v = X[:, i] 
        c = vec(H' \ v)  # least squares 
        w1 = D1 * c 
        w2 = D2 * c 
        ratio = norm(w2) / norm(w1) 
        
        # Compute the angle in degrees
        theta_rad = atan(norm(w2), norm(w1))
        theta_deg = theta_rad * (180 / π)
        push!(angles, theta_deg)
 
        # decision rule with an indecision zone 
        predicao = ratio > threshold ? digit_B : -1 
        if predicao == digit_B 
            acertos += 1 
        end 
    end 
    
    # Compute angle statistics
    media_angles = mean(angles)
     deviation_angles = std(angles)
    accuracy = acertos / total
 
    return acertos, accuracy, angles, media_angles,  deviation_angles
end

# Helper function to display the results in an organized way
function display_results(nome_digit, acertos, total, accuracy, media_angles,  deviation_angles)
    println("\n=== Results for digit $nome_digit ===")
    println("Correct: $acertos out of $total")
    println("Accuracy: $(round(accuracy * 100, digits=2))%")
    println("Mean of the angles: $(round(media_angles, digits=2))°")
    println("Standard deviation of the angles: $(round( deviation_angles, digits=2))°")
end

# Version that uses matrices A and B
function test_matrix_classification(A, B, digit_A, digit_B, D1, D2, H; threshold=1.0)
    println("Testing classification between digits $digit_A and $digit_B using matrices A and B")
    
    # Test matrix A
    acertos_A, accuracy_A, angles_A, media_A,  deviation_A = classify_all_A_matrix(A, digit_A, D1, D2, H; threshold=threshold)
    total_A = length(angles_A)
    display_results(digit_A, acertos_A, total_A, accuracy_A, media_A,  deviation_A)
    
    # Test matrix B  
    acertos_B, accuracy_B, angles_B, media_B,  deviation_B = classify_all_B_matrix(B, digit_B, D1, D2, H; threshold=threshold)
    total_B = length(angles_B)
    display_results(digit_B, acertos_B, total_B, accuracy_B, media_B,  deviation_B)
    
    # Accuracy geral
    accuracy_geral = (acertos_A + acertos_B) / (total_A + total_B)
    println("\n=== Overall Summary ===")
    println("Overall accuracy: $(round(accuracy_geral * 100, digits=2))%")
    
    return (angles_A, angles_B, accuracy_geral)
end

# Example of use:
angles_A, angles_B, accuracy = test_matrix_classification(A, B, digitA, digitB, D1_new, D2_our_new, H_new; threshold=1.0)

In [ ]:
p = plot_angle_histogram(angles_A, angles_B, 1, 5)
display(p)

In [ ]:
#classify_all_A(digitA, D1_new_reduzido, D2_our_new_reduzido; threshold=1.0)

In [ ]:
#classify_all_B(digitB, D1_new_reduzido, D2_our_new_reduzido; threshold=1.0)